In [1]:
#### Other libraries
import sys
import os

import ast
from collections import Counter

import re
import nltk

In [2]:
#### Pandas set-up
import numpy as np
import pandas as pd

In [3]:
#### Visualization
sys.path.append('/Users/debr/English-Homer') 
import matplotlib.pyplot as plt
import seaborn as sns
import bard_visualization as viz # My Vizualization library

Bard viz is on the stage!


In [4]:
# Import my functions
sys.path.append('/Users/debr/English-Homer/functions') 
import e_nlp as e

e enters! »----> use e.<function>

e.g., for pipeline:
		 nlp = e.NLPPipeline(language="english")

Stopwords customized:
  Added: {"'", 'three', 'two', 'five', 'four', 'ten', 'eight', "'and", 'seven', 'nine', 'n', 'six', 'one'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'\\', '“', '”', '’', '-', '—', '‘', '…'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
etymologypath = '/Users/debr/odysseys_en/word_ety_odysseys.tsv'
etymology_df = pd.read_csv(etymologypath, sep="\t", names=["word", "etymology"], encoding="utf-8")
etymology_df.sample(10, random_state=42)

,word,etymology
12507,scopes,['eng: scopes']
1119,bat,['fro: bast']
15537,underrate,['eng: under-']
16693,woodpile,['ang: wudu']
13855,stain,['enm: steinen']
361,aldermen,['eng: aldermen']
14676,tease,['ang: tæsan']
12949,shifts,['eng: shifts']
123,acquiring,['eng: acquiring']
2420,chimneys,['eng: chimneys']


In [32]:
etymology_df['ety_label'] = etymology_df['etymology'].str.extract(r"\['(\w{3})").astype("category")

In [33]:
print(etymology_df['ety_label'].sample(5, random_state=24))

6578     eng
8739     eng
5563     ang
10634    eng
6128     lat
Name: ety_label, dtype: category
Categories (44, object): ['afr', 'ang', 'ara', 'arg', ..., 'sux', 'tnq', 'tur', 'xno']


In [43]:
filepath = '/Users/debr/odysseys_en/Odyssey_Six_Modern.csv'
# Creating df and specific columns
odysseys_df = pd.read_csv(filepath, sep=",", encoding="utf-8")
odysseys_df = odysseys_df[['translator', 'book_num', 'tokens']]
odysseys_df['translator'] = odysseys_df['translator'].astype("category")
odysseys_df['tokens'] = odysseys_df['tokens'].apply(ast.literal_eval)
e.check_df(odysseys_df)

Oh, yeah! No missing values here!

* df columns: Index(['translator', 'book_num', 'tokens'], dtype='object') 

* Shape: (144, 3) 

* Total memory in MB: 2.860675


In [44]:
print(odysseys_df.memory_usage(deep=True)       / 1_000_000)
print(odysseys_df.memory_usage(deep=True).sum() / 1_000_000)

Index         0.000132
translator    0.000703
book_num      0.001152
tokens        2.858688
dtype: float64
2.860675


In [45]:
print(etymology_df.memory_usage(deep=True)       / 1_000_000)
print(etymology_df.memory_usage(deep=True).sum() / 1_000_000)

Index        0.000132
word         1.089155
etymology    1.256410
ety_label    0.020593
dtype: float64
2.36629


In [39]:
e.check_df(etymology_df)

Missing values in word
Missing values in ety_label

* df columns: Index(['word', 'etymology', 'ety_label'], dtype='object') 

* Shape: (16881, 3) 

* Total memory in MB: 2.36629


In [56]:
# Explode tokens column to have one word per row
odyssey_exploded = odysseys_df.explode("tokens")

# Merge with etymology_df to get ety_label
odyssey_exploded = odyssey_exploded\
                            .merge(etymology_df[['word', 'ety_label']], left_on='tokens', right_on='word', how='left')

# Group back by translator and book_num to form lists of ety_labels
odysseys_df['ety_labels'] = odyssey_exploded\
                            .groupby(['translator', 'book_num'],
                                      observed=True)['ety_label']\
                                    .apply(list).reset_index(drop=True)

odysseys_df.head(10)

,translator,book_num,tokens,ety_labels
0,AT_Murray,1,"[tell, muse, manysided, hero, traveled, far, w...","[ang, grc, eng, eng, eng, ang, enm, eng, fro, ..."
1,AT_Murray,2,"[child, morning, rosyfingered, dawn, appeared,...","[eng, enm, eng, eng, eng, eng, fra, eng, lat, ..."
2,AT_Murray,3,"[sun, rising, fair, sea, firmament, heaven, sh...","[ang, eng, lat, ang, eng, ang, enm, enm, eng, ..."
3,AT_Murray,4,"[reached, low, lying, city, lacedaemon, drove,...","[eng, ang, ang, lat, eng, eng, enm, eng, eng, ..."
4,AT_Murray,5,"[dawn, rose, couch, beside, tithonos, harbinge...","[eng, fra, lat, eng, eng, eng, enm, ang, eng, ..."
5,AT_Murray,6,"[odysseus, slept, overcome, sleep, toil, athen...","[eng, eng, ang, ang, enm, eng, eng, grc, lat, ..."
6,AT_Murray,7,"[thus, odysseus, wait, pray, girl, drove, town...","[enm, eng, enm, fro, eng, eng, eng, eng, ang, ..."
7,AT_Murray,8,"[child, morning, rosyfingered, dawn, appeared,...","[eng, enm, eng, eng, eng, eng, eng, fra, eng, ..."
8,AT_Murray,9,"[odysseus, answered, king, alkinoos, good, thi...","[eng, eng, enm, eng, enm, eng, ang, fra, lat, ..."
9,AT_Murray,10,"[thence, went, aeolian, island, lives, aeolus,...","[eng, eng, eng, eng, eng, eng, enm, eng, ang, ..."


In [59]:
from collections import Counter

odysseys_df['ety_label_counts'] = odysseys_df['ety_labels'].apply(lambda x: dict(Counter(x)))
odysseys_df['ety_label_counts'][0]

{'ang': 459,
 'grc': 12,
 'eng': 804,
 'enm': 294,
 'fro': 45,
 'lat': 113,
 'xno': 20,
 'non': 25,
 'heb': 7,
 'fra': 12,
 'p_g': 3,
 'ita': 2,
 'tur': 1,
 'msa': 1,
 'gml': 1,
 'spa': 1}

In [58]:
set(odysseys_df['ety_labels'][0])

{'ang',
 'eng',
 'enm',
 'fra',
 'fro',
 'gml',
 'grc',
 'heb',
 'ita',
 'lat',
 'msa',
 'non',
 'p_g',
 'spa',
 'tur',
 'xno'}